# Baseline de Contexto para Arquivos DXF

Este notebook estabelece um baseline técnico mínimo para o problema atual do projeto: **entender quanto da estrutura do DXF o agente consegue acessar e quanto contexto é consumido pela estratégia atual**.

O foco é responder a quatro perguntas:

1. Quantas entidades raiz existem e quanto a geometria cresce após a decomposição?
2. O `CadModel` preserva adequadamente as entidades raiz necessárias ao agente?
3. Quais informações ficam fora do índice ou não são diretamente endereçáveis?
4. Quanto contexto é introduzido pelo resumo estático e por tools globais?

## Conceitos Usados no Notebook

- **Entidade raiz:** entidade persistente diretamente em `doc.modelspace()`.
- **INSERT / bloco:** referência a uma definição de bloco que pode encapsular várias geometrias.
- **Componente expandido:** componente observado após decomposição recursiva de uma entidade raiz.
- **Handle:** identificador persistente de uma entidade DXF; é a principal referência para rastreabilidade.
- **Layer:** camada declarada da entidade; é um sinal estrutural útil, mas não deve ser tratada como ontologia.
- **Proveniência:** relação entre um componente observado e a entidade raiz persistente que o originou.

## Exemplo de Leitura do DXF

```text
Modelspace
├── LINE                     handle=10   layer=PAREDES
├── MTEXT                    handle=11   layer=TEXTOS
└── INSERT                   handle=12   layer=MOBILIARIO
    └── bloco "MESA_01"
        ├── LINE
        ├── LINE
        └── CIRCLE
```

Neste exemplo, temos que:
- LINE, MTEXT e INSERT são entidades raiz, pois existem diretamente no Modelspace
- O INSERT de handle 12 referencia a definição do bloco MESA_01
- Antes da expansão, o agente enxerga o mobiliário "MESA_01" como uma única entidade raiz (INSERT)
- Após a decomposição recursiva, as duas LINEs e o CIRCLE internos passam a aparecer como "componentes expandidos"
- Esses componentes expandidos podem não possuir handle próprio. Por isso, para manter a proveniência, precisamos saber a origem deles
- O layer=MOBILIARIO funciona como um sinal sobre o INSERT, mas não garante que o elemento represente semanticamente uma mesa
- A geometria estaria um nível abaixo dos componentes expandidos do bloco

## Fluxo Atual da Aplicação

```text
DXF -> ezdxf -> Drawing -> CadModel em memória
                              |
                              +-> describe_model() -> resumo no system prompt --+
                              |                                                 |
                              +-> tools -> JSON sob demanda --------------------+-> LLM
                              |                                                 |
                              +-> prims_payload() -> viewer, fora do contexto   |
                                                                                |
instruções + schemas das tools + usuário + histórico ---------------------------+
```

O DXF completo e o payload do viewer não entram diretamente na janela do modelo. O custo estático vem principalmente das instruções, schemas e `describe_model()`. O custo dinâmico cresce com as mensagens e, especialmente, com os resultados das tools acumulados na conversa.

## 1. Setup

In [1]:
import logging
import os
import sys
import time
from collections import Counter
from pathlib import Path
from typing import Any

import ezdxf
import pandas as pd
from ezdxf.disassemble import recursive_decompose

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "cad" / "model.py").exists():
            return candidate
    raise RuntimeError(
        "Não foi possível localizar o repositório. "
        "Abra o notebook dentro do projeto ou defina REPO_ROOT manualmente."
    )

REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data/dxf"
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# O kernel abre com o cwd do proprio notebook, em experiments/, entao qualquer
# import de `src` precisa vir depois da linha acima.
from src.cad.loader import load_dxf  # noqa: E402
from src.cad.model import build_model  # noqa: E402

def safe_handle(entity) -> str | None:
    value = entity.dxf.get("handle", None)
    return str(value) if value else None

def safe_layer(entity) -> str:
    return str(entity.dxf.get("layer", "0"))

def safe_block_name(entity) -> str | None:
    if entity.dxftype() != "INSERT":
        return None
    value = entity.dxf.get("name", None)
    return str(value) if value else None

logging.getLogger("ezdxf").addFilter(
    lambda record: "copy process ignored" not in record.getMessage()
)

print("REPO_ROOT:", REPO_ROOT)
print("DATA_DIR:", DATA_DIR)

REPO_ROOT: /home/lbandeira/projects/ict.h3.dxf-view
DATA_DIR: /home/lbandeira/projects/ict.h3.dxf-view/data/dxf


### Arquivos Avaliados

Usamos apenas dois cenários para manter o baseline simples:

- `001_room_only.dxf`: caso pequeno/controlável;
- `003_complete_floor.dxf`: caso maior e estruturalmente mais complexo.

In [2]:
all_paths = sorted(DATA_DIR.glob("*.dxf"))
if not all_paths:
    raise FileNotFoundError(f"Nenhum .dxf encontrado em {DATA_DIR}")

preferred_names = ["001_room_only.dxf", "003_complete_floor.dxf"]
DXF_PATHS = [DATA_DIR / name for name in preferred_names if (DATA_DIR / name).exists()]

print(f"Python: {sys.version.split()[0]} | ezdxf: {ezdxf.__version__}")
for path in DXF_PATHS:
    print("-", path.name, f"({path.stat().st_size / 1024**2:.2f} MiB)")

Python: 3.11.15 | ezdxf: 1.4.4
- 001_room_only.dxf (1.41 MiB)
- 003_complete_floor.dxf (13.94 MiB)


## 2. Verifica a Estrutura do DXF

In [3]:
DOCS: dict[str, Any] = {}
LOAD_WARNINGS: dict[str, list[str]] = {}
TOP_LEVEL: dict[str, pd.DataFrame] = {}
EXPANSION: dict[str, pd.DataFrame] = {}

for path in DXF_PATHS:
    doc, warnings = load_dxf(path.read_bytes())
    DOCS[path.name] = doc
    LOAD_WARNINGS[path.name] = list(warnings)

    root_rows = []
    expansion_rows = []

    for entity in doc.modelspace():
        root_handle = safe_handle(entity)
        root_rows.append({
            "handle": root_handle,
            "type": entity.dxftype(),
            "layer": safe_layer(entity),
            "block": safe_block_name(entity),
        })

        try:
            leaves = list(recursive_decompose([entity]))
            error = None
        except Exception as exc:
            leaves = [entity]
            error = repr(exc)

        expansion_rows.append({
            "parent_handle": root_handle,
            "parent_type": entity.dxftype(),
            "parent_layer": safe_layer(entity),
            "parent_block": safe_block_name(entity),
            "expanded_count": len(leaves),
            "expanded_with_handle": sum(safe_handle(leaf) is not None for leaf in leaves),
            "decompose_error": error,
        })

    TOP_LEVEL[path.name] = pd.DataFrame(root_rows)
    EXPANSION[path.name] = pd.DataFrame(expansion_rows)

structure_rows = []
for name in DOCS:
    roots = TOP_LEVEL[name]
    expansion = EXPANSION[name]
    expanded_total = int(expansion["expanded_count"].sum())

    structure_rows.append({
        "arquivo": name,
        "entidades_raiz": len(roots),
        "tipos_raiz": roots["type"].nunique(),
        "layers_usados_raiz": roots["layer"].nunique(),
        "raizes_sem_handle": int(roots["handle"].isna().sum()),
        "componentes_expandidos": expanded_total,
        "fator_expansao": expanded_total / max(1, len(roots)),
        "max_componentes_em_um_pai": int(expansion["expanded_count"].max()),
        "erros_decomposicao": int(expansion["decompose_error"].notna().sum()),
    })

structure_df = pd.DataFrame(structure_rows)
display(structure_df)

for name, expansion in EXPANSION.items():
    print(f"\nMaiores entidades compostas — {name}")
    display(
        expansion.sort_values("expanded_count", ascending=False)
        [["parent_handle", "parent_type", "parent_layer", "parent_block", "expanded_count"]]
        .head(5)
    )

,arquivo,entidades_raiz,tipos_raiz,layers_usados_raiz,raizes_sem_handle,componentes_expandidos,fator_expansao,max_componentes_em_um_pai,erros_decomposicao
0,001_room_only.dxf,70,7,19,0,782,11.171,244,0
1,003_complete_floor.dxf,4460,10,65,0,70801,15.875,3056,0



Maiores entidades compostas — 001_room_only.dxf


,parent_handle,parent_type,parent_layer,parent_block,expanded_count
11,E0F08AA,INSERT,M095-21-PBAQ-0001-P-2S-01$0$ARQ-HOS,M095-21-PBAQ-0001-P-2S-01$0$E018b-p,244
8,E0F0843,INSERT,M095-21-PBAQ-0001-P-2S-01$0$ARQ-EQP-HIDE-CIVIL,M095-21-PBAQ-0001-P-2S-01$0$M091-13_XREF$0$Tel...,233
2,E0F0143,INSERT,M095-21-PBAQ-0001-P-2S-01$0$ARQ-MOB,M095-21-PBAQ-0001-P-2S-01$0$E010-18-BS-MOBILIA...,66
0,E0EFBD5,INSERT,M095-21-PBAQ-0001-P-2S-01$0$ARQ-BAN,M095-21-PBAQ-0001-P-2S-01$0$BA_01_MA_01,48
37,E0F0B30,INSERT,M095-21-PBAQ-0001-P-2S-01$0$ARQ-EQP,M095-21-PBAQ-0001-P-2S-01$0$EQL_MD_monitor lcd...,24



Maiores entidades compostas — 003_complete_floor.dxf


,parent_handle,parent_type,parent_layer,parent_block,expanded_count
785,E0EFA28,INSERT,M095-21-PBAQ-0001-P-2S-01$0$ARQ-BASE,M095-21-PBAQ-0001-P-2S-01$0$A$C269d95ed,3056
772,E0EFA05,INSERT,M095-21-PBAQ-0001-P-2S-01$0$ARQ-BAN,M095-21-PBAQ-0001-P-2S-01$0$Bancada_Expurgo_01,2316
4349,E0F1507,INSERT,M095-21-PBAQ-0001-P-2S-01$0$ARQ-MOB,M095-21-PBAQ-0001-P-2S-01$0$A$C1ed5ab4b,786
4350,E0F1508,INSERT,M095-21-PBAQ-0001-P-2S-01$0$ARQ-VEG,M095-21-PBAQ-0001-P-2S-01$0$arv-12,584
4344,E0F14F6,INSERT,M095-21-PBAQ-0001-P-2S-01$0$ARQ-VEG,M095-21-PBAQ-0001-P-2S-01$0$arv-12,584


A comparação importante é entre **entidades raiz** e **componentes expandidos**. Um fator de expansão alto indica que uma representação baseada somente nas entidades raiz é compacta, mas pode esconder granularidade relevante para consulta espacial ou identificação de elementos.

Os cinco maiores pais mostram o extremo; a tabela de concentração mostra a forma. O que importa não é a média, e sim que a massa de geometria se acumula em poucas entidades compostas — e cada uma delas é um id só. Um bloco de mobiliário com centenas de componentes é semanticamente um elemento; um bloco que encapsula um ambiente inteiro, com paredes e portas dentro, é o caso oposto. A granularidade estrutural do DXF não coincide com a semântica, e varia por bloco.

In [4]:
concentration_rows = []
for name, expansion in EXPANSION.items():
    ordered = expansion["expanded_count"].sort_values(ascending=False).reset_index(drop=True)
    total = int(ordered.sum())
    row = {"arquivo": name, "entidades_raiz": len(ordered), "componentes": total}
    for n in (1, 10, 100, 500):
        # Quando n excede o numero de raizes, o recorte e o desenho inteiro.
        row[f"top_{n}_pct"] = 100 * int(ordered.head(n).sum()) / max(1, total)
    concentration_rows.append(row)

concentration_df = pd.DataFrame(concentration_rows)
display(concentration_df)

,arquivo,entidades_raiz,componentes,top_1_pct,top_10_pct,top_100_pct,top_500_pct
0,001_room_only.dxf,70,782,31.202,90.793,100.000,100.000
1,003_complete_floor.dxf,4460,70801,4.316,13.561,52.488,89.373


## 3. Verifica o que o `CadModel` Consegue Endereçar

In [5]:
MODELS: dict[str, Any] = {}

for name, doc in DOCS.items():
    MODELS[name] = build_model(
        doc,
        filename=name,
        warnings=LOAD_WARNINGS[name],
    )

parser_rows = []
for name, model in MODELS.items():
    raw = TOP_LEVEL[name]
    raw_handles = set(raw["handle"].dropna())
    indexed_handles = set(model.by_id)

    parser_rows.append({
        "arquivo": name,
        "entidades_raiz": len(raw),
        "cadmodel_entities": len(model.entities),
        "handle_coverage_pct": 100 * len(raw_handles & indexed_handles) / max(1, len(raw_handles)),
        "types_match": Counter(raw["type"]) == Counter(e.type for e in model.entities),
        "layers_match": Counter(raw["layer"]) == Counter(e.layer for e in model.entities),
        "without_prims": sum(len(e.prims) == 0 for e in model.entities),
        "without_bbox": sum(e.bbox is None for e in model.entities),
        "model_warnings": len(model.warnings),
    })

parser_df = pd.DataFrame(parser_rows)
display(parser_df)

,arquivo,entidades_raiz,cadmodel_entities,handle_coverage_pct,types_match,layers_match,without_prims,without_bbox,model_warnings
0,001_room_only.dxf,70,70,100.000,True,True,0,0,0
1,003_complete_floor.dxf,4460,4460,100.000,True,True,0,0,0


`without_prims`, `without_bbox` e `model_warnings` confirmam que toda entidade produziu geometria, mas não que a geometria está correta. Um contorno marcado como fechado pode colapsar para menos de 3 pontos durante o achatamento — a tolerância é proporcional à extensão do desenho — e `polygon_area` devolve zero para ele.

Isso aparece de duas formas:

- **visível:** todos os contornos fechados da entidade colapsaram, e a `area` sai zero;
- **silenciosa:** apenas parte colapsou, e a entidade reporta uma área plausível, porém menor que a real.

A segunda é a que importa: a entidade tem primitiva, tem bbox e tem medida preenchida, então passa por todas as verificações da tabela anterior. É o limite do "suficientemente" da conclusão 1.

In [6]:
from src.cad.geometry import PolylinePrim

collapse_rows = []
for name, model in MODELS.items():
    afetadas = contornos = silenciosa = 0

    for entity in model.entities:
        fechadas = [
            prim
            for prim in entity.prims
            if isinstance(prim, PolylinePrim) and prim.closed
        ]
        # polygon_area devolve 0 com menos de 3 pontos: o contorno colapsou.
        colapsados = sum(len(prim.pts) < 3 for prim in fechadas)
        if not colapsados:
            continue

        afetadas += 1
        contornos += colapsados
        if entity.area:
            silenciosa += 1

    collapse_rows.append({
        "arquivo": name,
        "entidades_afetadas": afetadas,
        "contornos_colapsados": contornos,
        "perda_visivel_area_zero": afetadas - silenciosa,
        "perda_silenciosa_area_positiva": silenciosa,
    })

collapse_df = pd.DataFrame(collapse_rows)
display(collapse_df)

,arquivo,entidades_afetadas,contornos_colapsados,perda_visivel_area_zero,perda_silenciosa_area_positiva
0,001_room_only.dxf,0,0,0,0
1,003_complete_floor.dxf,65,232,10,55


### Limites Atuais

Duas fronteiras são especialmente relevantes para o Context Manager a ser desenvolvido:

1. `build_model()` indexa o **Modelspace**; conteúdo exclusivo de Paperspace não participa do índice usado pelo agente.
2. A decomposição pode produzir componentes sem identificador persistente próprio. Para manter rastreabilidade, consultas sobre componentes internos precisam preservar a relação com a entidade raiz que os originou.

In [7]:
coverage_rows = []

for name, doc in DOCS.items():
    model = MODELS[name]
    paperspace_count = sum(
        len(layout)
        for layout in doc.layouts
        if layout.name.lower() != "model"
    )

    expansion = EXPANSION[name]
    expanded_total = int(expansion["expanded_count"].sum())
    expanded_with_handle = int(expansion["expanded_with_handle"].sum())

    coverage_rows.append({
        "arquivo": name,
        "modelspace_indexado": len(model.entities),
        "paperspace_fora_do_indice": paperspace_count,
        "componentes_expandidos": expanded_total,
        "componentes_expandidos_com_handle_pct": (
            100 * expanded_with_handle / max(1, expanded_total)
        ),
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

,arquivo,modelspace_indexado,paperspace_fora_do_indice,componentes_expandidos,componentes_expandidos_com_handle_pct
0,001_room_only.dxf,70,2,782,7.161
1,003_complete_floor.dxf,4460,18,70801,5.041


## 4. Verifica o Contexto Estático do LLM

In [8]:
from src.ai_modules.tools import CAD_TOOLS
from src.cad.model import describe_model

instructions_path = (
    REPO_ROOT / "src" / "ai_modules" / "agents" / "cad_agent" / "instructions.md"
)
instructions_chars = len(instructions_path.read_text(encoding="utf-8"))

static_rows = []
for name, model in MODELS.items():
    summary = describe_model(model)

    ranked_layers = sorted(
        model.layers,
        key=lambda layer: (-layer.count, layer.name.lower()),
    )
    visible_layers = ranked_layers[:25]
    visible_entities = sum(layer.count for layer in visible_layers)
    total_entities = sum(layer.count for layer in model.layers)

    static_rows.append({
        "arquivo": name,
        "instrucoes_chars": instructions_chars,
        "tools_expostas": len(CAD_TOOLS),
        "describe_model_chars": len(summary),
        "layers_total": len(model.layers),
        "layers_no_resumo": len(visible_layers),
        "entidades_cobertas_pelos_layers_visiveis_pct": (
            100 * visible_entities / max(1, total_entities)
        ),
        "blocos_inseridos_distintos": len(model.insert_counts),
        "blocos_no_resumo": min(15, len(model.insert_counts)),
    })

static_context_df = pd.DataFrame(static_rows)
display(static_context_df)

,arquivo,instrucoes_chars,tools_expostas,describe_model_chars,layers_total,layers_no_resumo,entidades_cobertas_pelos_layers_visiveis_pct,blocos_inseridos_distintos,blocos_no_resumo
0,001_room_only.dxf,2620,12,2624,31,25,100.000,13,13
1,003_complete_floor.dxf,2620,12,3124,119,25,93.789,204,15


## 5. Verifica o Impacto de Tools no Contexto

Mede quanto contexto cada tipo de pergunta acrescenta à conversa quando o acesso ao desenho é global. É a linha de base contra a qual o notebook 2 vai comparar a
recuperação seletiva.

| Caso | O que estressa |
| --- | --- |
| `entidade_pontual` | piso de custo: um id conhecido, resposta curta |
| `contagem_por_atributo` | identidade por nome de bloco ou layer |
| `enumeracao_ambientes` | acúmulo da saída de várias tools |
| `espacial` | localização por coordenada, derivada da extensão |
| `resumo_interface` | query de exemplo da interface |
| `layers_ranking` | listagem ampla ordenada |

**Por que dois modelos?**

A mesma pergunta produz estratégias diferentes: em traces reais, o Sonnet responde "faça um resumo" com quatro tool calls e o prompt crescendo de 5.941 para 12.382 tokens, enquanto o Haiku responde a mesma coisa numa única requisição, sem chamar tool alguma — direto do resumo estático. O custo de acesso, portanto, não é propriedade só da pergunta: depende do modelo. São 2 modelos × 6 casos × 2 arquivos = 24 runs.

### Como Ler as Métricas de Token

`delta_prompt` é a diferença entre o prompt observado na primeira e na última requisição, e **não equivale ao que as tools inseriram**: com raciocínio estendido ligado, os blocos de pensamento são reecoados a cada requisição, e o cache move tokens entre `input`, `cache_read` e `cache_write`. Por isso os componentes são gravados separadamente — sem eles seria fácil atribuir às tools um custo que é do modo de raciocínio. Em run de requisição única não há crescimento, e o delta é zero.

In [10]:
from agno.db.in_memory import InMemoryDb

from src.ai_modules.agents import get_cad_agent
from src.ai_modules.llm_settings import _gateway_model
from src.ai_modules.runner import run_chat
from src.ai_modules.tools import UI_TOOL_NAMES
from src.api.core.config import require_ai_gateway
from src.api.models.chat import ChatRequest

RUN_TOOL_IMPACT = True

TOOL_IMPACT_MODELS = [
    {"rotulo": "haiku-4.5", "model_id": "claude-haiku-4-5"},
    {"rotulo": "sonnet-4.6", "model_id": "claude-sonnet-4-6"},
]

TOOL_IMPACT_PROBES = [
    {
        "caso": "entidade_pontual",
        "pergunta": "Quais são as dimensões da entidade {entity_id}?",
    },
    {
        "caso": "contagem_por_atributo",
        "pergunta": "Quantas portas existem neste desenho?",
    },
    {
        "caso": "enumeracao_ambientes",
        "pergunta": "Liste os ambientes identificados no desenho e diga quantos são.",
    },
    {
        "caso": "espacial",
        "pergunta": "O que existe no quadrante superior esquerdo do desenho?",
    },
    {
        "caso": "resumo_interface",
        "pergunta": "Faça um resumo do que existe neste desenho.",
    },
    {
        "caso": "layers_ranking",
        "pergunta": "Quais são os layers com mais entidades?",
    },
]

def probe_entity_id(cad_model: Any) -> str:
    """Entidade de maior area, para o caso pontual ser estavel entre execucoes."""
    return max(cad_model.entities, key=lambda e: (e.area or 0.0, e.id)).id

def observed_prompt_tokens(request: dict[str, Any]) -> int:
    """Ocupacao da janela numa requisicao: o que foi lido do cache tambem ocupa."""
    return sum(
        int(request.get(field) or 0)
        for field in ("input_tokens", "cache_read_tokens", "cache_write_tokens")
    )

def total_of(requests: list[dict[str, Any]], field: str) -> int:
    return sum(int(request.get(field) or 0) for request in requests)

async def observe_tool_impact(
    filename: str,
    cad_model: Any,
    probe: dict[str, str],
    model_profile: dict[str, str],
) -> dict[str, Any]:
    pergunta = probe["pergunta"].replace("{entity_id}", probe_entity_id(cad_model))

    llm_model = _gateway_model(model_profile["model_id"], thinking=True)
    agent = await get_cad_agent(user_id="eda-local", model=cad_model, db=InMemoryDb())
    agent.model = llm_model
    agent.debug_mode = False

    requests = []
    tool_calls = []
    error = None
    started = time.perf_counter()

    try:
        stream = await run_chat(
            agent,
            ChatRequest(
                message=pergunta,
                chat_id=(
                    f"eda-tool-{Path(filename).stem}-{probe['caso']}-"
                    f"{model_profile['rotulo']}"
                ),
                document_id=filename,
            ),
            "eda-local",
            cad_model,
        )

        async for event in stream:
            event_name = type(event).__name__

            if event_name == "ModelRequestCompletedEvent":
                requests.append({
                    "input_tokens": event.input_tokens,
                    "output_tokens": event.output_tokens,
                    "cache_read_tokens": event.cache_read_tokens,
                    "cache_write_tokens": event.cache_write_tokens,
                    "reasoning_tokens": event.reasoning_tokens,
                })

            elif event_name == "ToolCallCompletedEvent" and event.tool:
                result = event.content if event.content is not None else event.tool.result
                tool_calls.append({
                    "name": event.tool.tool_name,
                    "result_chars": len(str(result or "")),
                })

            elif event_name == "RunErrorEvent":
                error = str(getattr(event, "error", "erro não detalhado"))

    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"

    # Sem segunda requisicao nao houve crescimento: o delta e zero, nao indefinido.
    prompt_primeiro = observed_prompt_tokens(requests[0]) if requests else None
    prompt_ultimo = observed_prompt_tokens(requests[-1]) if requests else None
    delta = (prompt_ultimo - prompt_primeiro) if requests else None

    consulta = [c for c in tool_calls if c["name"] not in UI_TOOL_NAMES]
    interface = [c for c in tool_calls if c["name"] in UI_TOOL_NAMES]

    return {
        "modelo": model_profile["rotulo"],
        "arquivo": filename,
        "caso": probe["caso"],
        "tool_calls_consulta": len(consulta),
        "tool_calls_interface": len(interface),
        "tools_observadas": [c["name"] for c in tool_calls],
        "model_requests": len(requests),
        "tool_output_chars": sum(c["result_chars"] for c in consulta),
        "prompt_primeiro": prompt_primeiro,
        "prompt_ultimo": prompt_ultimo,
        "delta_prompt": delta,
        "input_tokens": total_of(requests, "input_tokens"),
        "cache_read_tokens": total_of(requests, "cache_read_tokens"),
        "cache_write_tokens": total_of(requests, "cache_write_tokens"),
        "reasoning_tokens": total_of(requests, "reasoning_tokens"),
        "output_tokens": total_of(requests, "output_tokens"),
        "latencia_s": time.perf_counter() - started,
        "erro": error,
    }

TOOL_IMPACT_RESULTS = []

if not RUN_TOOL_IMPACT:
    total = len(TOOL_IMPACT_MODELS) * len(MODELS) * len(TOOL_IMPACT_PROBES)
    print(f"Experimento desativado. Defina RUN_TOOL_IMPACT = True para executar os {total} runs.")
else:
    require_ai_gateway()

    total_runs = len(TOOL_IMPACT_MODELS) * len(MODELS) * len(TOOL_IMPACT_PROBES)
    run_number = 0

    for model_profile in TOOL_IMPACT_MODELS:
        for filename, cad_model in MODELS.items():
            for probe in TOOL_IMPACT_PROBES:
                run_number += 1
                print(
                    f"[{run_number:02d}/{total_runs}] {model_profile['rotulo']} | "
                    f"{filename} | {probe['caso']}"
                )
                TOOL_IMPACT_RESULTS.append(
                    await observe_tool_impact(filename, cad_model, probe, model_profile)
                )

if TOOL_IMPACT_RESULTS:
    tool_impact_df = pd.DataFrame(TOOL_IMPACT_RESULTS)

    # 1. Estrategia de acesso: quantas tools de consulta cada pergunta aciona.
    acesso_df = (
        tool_impact_df.groupby(["caso", "modelo"], sort=False)
        .agg(
            runs=("arquivo", "size"),
            tool_calls_consulta=("tool_calls_consulta", "median"),
            tool_calls_interface=("tool_calls_interface", "median"),
            requests=("model_requests", "median"),
            tool_output_chars=("tool_output_chars", "median"),
            latencia_s=("latencia_s", "mean"),
            erros=("erro", lambda v: v.notna().sum()),
        )
        .reset_index()
        .sort_values(["caso", "modelo"])
        .round(2)
    )
    display(acesso_df)

    # 2. De onde vem o token: o delta so e atribuivel as tools depois de separar
    #    o raciocinio reecoado e o que o cache move entre os campos.
    tokens_df = (
        tool_impact_df.groupby(["caso", "modelo"], sort=False)
        .agg(
            prompt_primeiro=("prompt_primeiro", "median"),
            prompt_ultimo=("prompt_ultimo", "median"),
            delta_prompt=("delta_prompt", "median"),
            input_tokens=("input_tokens", "median"),
            cache_read=("cache_read_tokens", "median"),
            cache_write=("cache_write_tokens", "median"),
            reasoning=("reasoning_tokens", "median"),
            output=("output_tokens", "median"),
        )
        .reset_index()
        .sort_values("delta_prompt", ascending=False)
        .round(1)
    )
    display(tokens_df)

[01/24] haiku-4.5 | 001_room_only.dxf | entidade_pontual
[02/24] haiku-4.5 | 001_room_only.dxf | contagem_por_atributo
[03/24] haiku-4.5 | 001_room_only.dxf | enumeracao_ambientes
[04/24] haiku-4.5 | 001_room_only.dxf | espacial
[05/24] haiku-4.5 | 001_room_only.dxf | resumo_interface
[06/24] haiku-4.5 | 001_room_only.dxf | layers_ranking
[07/24] haiku-4.5 | 003_complete_floor.dxf | entidade_pontual
[08/24] haiku-4.5 | 003_complete_floor.dxf | contagem_por_atributo


WARNING  Could not run function highlight_entities(ids=query_layer_ARQ-POR_INSERT, label=Portas (69), zoom=True): 1
         validation error for highlight_entities                                                                   
         ids                                                                                                       
           Input should be a valid list [type=list_type, input_value='query_layer_ARQ-POR_INSERT', input_type=str] 
             For further information visit https://errors.pydantic.dev/2.13/v/list_type

ERROR    1 validation error for highlight_entities                                                                 
         ids                                                                                                       
           Input should be a valid list [type=list_type, input_value='query_layer_ARQ-POR_INSERT', input_type=str] 
             For further information visit https://errors.pydantic.dev/2.13/v/list_type                            
         Traceback (most recent call last):                                                                        
           File                                                                                                    
         "/home/lbandeira/projects/ict.h3.dxf-view/.venv/lib/python3.11/site-packages/agno/tools/function.py", line
         1090, in execute                                                                                          
             result = self.function.entrypoint(**entrypoint_args, **self.arguments)                                
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                
           File                                                                                                    
         "/home/lbandeira/projects/ict.h3.dxf-view/.venv/lib/python3.11/site-packages/pydantic/_internal/_validate_
         call.py", line 40, in wrapper_function                                                                    
             return wrapper(*args, **kwargs)                                                                       
                    ^^^^^^^^^^^^^^^^^^^^^^^^                                                                       
           File                                                                                                    
         "/home/lbandeira/projects/ict.h3.dxf-view/.venv/lib/python3.11/site-packages/pydantic/_internal/_validate_
         call.py", line 137, in __call__                                                                           
             res = self.__pydantic_validator__.validate_python(pydantic_core.ArgsKwargs(args, kwargs))             
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^             
         pydantic_core._pydantic_core.ValidationError: 1 validation error for highlight_entities                   
         ids                                                                                                       
           Input should be a valid list [type=list_type, input_value='query_layer_ARQ-POR_INSERT', input_type=str] 
             For further information visit https://errors.pydantic.dev/2.13/v/list_type

[09/24] haiku-4.5 | 003_complete_floor.dxf | enumeracao_ambientes
[10/24] haiku-4.5 | 003_complete_floor.dxf | espacial
[11/24] haiku-4.5 | 003_complete_floor.dxf | resumo_interface
[12/24] haiku-4.5 | 003_complete_floor.dxf | layers_ranking
[13/24] sonnet-4.6 | 001_room_only.dxf | entidade_pontual
[14/24] sonnet-4.6 | 001_room_only.dxf | contagem_por_atributo
[15/24] sonnet-4.6 | 001_room_only.dxf | enumeracao_ambientes
[16/24] sonnet-4.6 | 001_room_only.dxf | espacial
[17/24] sonnet-4.6 | 001_room_only.dxf | resumo_interface
[18/24] sonnet-4.6 | 001_room_only.dxf | layers_ranking
[19/24] sonnet-4.6 | 003_complete_floor.dxf | entidade_pontual
[20/24] sonnet-4.6 | 003_complete_floor.dxf | contagem_por_atributo
[21/24] sonnet-4.6 | 003_complete_floor.dxf | enumeracao_ambientes
[22/24] sonnet-4.6 | 003_complete_floor.dxf | espacial
[23/24] sonnet-4.6 | 003_complete_floor.dxf | resumo_interface
[24/24] sonnet-4.6 | 003_complete_floor.dxf | layers_ranking


,caso,modelo,runs,tool_calls_consulta,tool_calls_interface,requests,tool_output_chars,latencia_s,erros
1,contagem_por_atributo,haiku-4.5,2,2.500,1.000,4.000,191.000,17.160,0
7,contagem_por_atributo,sonnet-4.6,2,2.000,0.500,2.500,123.500,19.690,0
0,entidade_pontual,haiku-4.5,2,1.000,0.000,2.000,60.000,5.810,0
6,entidade_pontual,sonnet-4.6,2,1.000,1.000,3.000,60.000,10.300,0
2,enumeracao_ambientes,haiku-4.5,2,1.500,1.000,3.000,107.000,14.470,0
8,enumeracao_ambientes,sonnet-4.6,2,2.000,1.000,3.500,131.500,51.300,0
3,espacial,haiku-4.5,2,1.000,1.000,3.000,61.000,16.720,0
9,espacial,sonnet-4.6,2,6.500,1.500,6.000,339.500,120.790,0
5,layers_ranking,haiku-4.5,2,0.000,0.000,1.000,0.000,8.000,0
11,layers_ranking,sonnet-4.6,2,0.000,0.000,1.000,0.000,13.200,0


,caso,modelo,prompt_primeiro,prompt_ultimo,delta_prompt,input_tokens,cache_read,cache_write,reasoning,output
8,enumeracao_ambientes,sonnet-4.6,"5,901.000","25,078.000","19,177.000","68,509.500",0.000,0.000,567.500,"2,970.500"
10,resumo_interface,sonnet-4.6,"5,899.000","17,553.500","11,654.500","52,684.000",0.000,0.000,570.500,"2,670.000"
9,espacial,sonnet-4.6,"5,900.000","16,154.000","10,254.000","77,842.000",0.000,0.000,"1,621.500","5,226.500"
1,contagem_por_atributo,haiku-4.5,"5,942.000","11,077.500","5,135.500","34,035.500",0.000,0.000,455.000,"1,367.500"
3,espacial,haiku-4.5,"5,945.000","9,883.000","3,938.000","25,219.000",0.000,0.000,456.000,"1,200.000"
2,enumeracao_ambientes,haiku-4.5,"5,946.000","8,552.500","2,606.500","22,559.000",0.000,0.000,328.000,924.000
7,contagem_por_atributo,sonnet-4.6,"5,897.000","6,755.000",858.000,"16,025.500",0.000,0.000,202.500,728.500
6,entidade_pontual,sonnet-4.6,"5,902.000","6,557.000",655.000,"18,817.000",0.000,0.000,80.000,492.500
0,entidade_pontual,haiku-4.5,"5,947.000","6,373.000",426.000,"12,320.000",0.000,0.000,89.000,370.000
5,layers_ranking,haiku-4.5,"5,941.000","5,941.000",0.000,"5,941.000",0.000,0.000,116.000,754.500


## 6. Análise

### Resultado 1 — o parser preserva as entidades raiz

Os dois arquivos carregaram sem avisos. O CadModel reproduziu 100% dos handles, as mesmas contagens de tipo e de layer do modelspace, e nenhuma entidade ficou sem primitiva ou sem *bounding box*. Como baseline de representação, a camada raiz é confiável.

<!-- A ressalva está nos contornos colapsados. No 003_complete_floor, 65 entidades somam 232 contornos fechados que colapsaram para menos de 3 pontos durante o achatamento. Dessas, 10 reportam área zero e 55 reportam uma área plausível, porém menor que a real. Essas 55 passam por todas as verificações da seção 3: têm primitiva, têm bbox e têm medida preenchida. O 001_room_only não apresenta o efeito, o que é coerente com uma tolerância de achatamento proporcional à extensão do desenho: quanto maior o desenho, mais grosseira a curva. -->

### Resultado 2 — a geometria relevante está encapsulada e não é diretamente endereçável

A decomposição recursiva leva 70 e 4.460 entidades raiz a 782 e 70.801 componentes. Como as sub-entidades produzidas na decomposição não têm handle, apenas 7,2% e 5,0% dos componentes possuem identificador próprio. O percentual isolado engana, porque a maioria das raízes expande para um único componente e é perfeitamente endereçável.

A concentração mostra a forma real: no 003_complete_floor, as 100 maiores raízes concentram 52,5% da geometria, e as 500 maiores, 89,4%; um único INSERT chega a 3.056 componentes. No 001_room_only, uma só entidade concentra 31,2%. Um bloco de mobiliário com centenas de componentes é semanticamente um elemento — expandí-lo seria ruído. Um bloco que encapsula um ambiente inteiro, com paredes e portas dentro, é o oposto: hoje não existe id para a parede. A granularidade estrutural do DXF não coincide com a semântica, e varia de bloco para bloco. Nenhum nível fixo de endereço resolve os dois casos.

### Resultado 3 — o custo de contexto depende do modelo, e não vem do payload das tools

Duas conclusões, e a segunda contraria a formulação original deste achado.

O custo é propriedade do par pergunta-modelo. Todo run parte de um piso de aproximadamente 5.900 tokens de contexto estático — instruções, schemas das 12 tools e
describe_model() — antes de qualquer pergunta. A partir daí, a mesma pergunta diverge:

| Caso | Haiku 4.5 | Sonnet 4.6 |
| --- | --- | --- |
| `resumo_interface` | 0 tool calls, delta 0 | 6 tool calls, delta 11.654 |
| `espacial` | 1 tool call, delta 3.938 | 6,5 tool calls, delta 10.254 |
| `enumeracao_ambientes` | 1,5 tool calls, delta 2.606 | 2 tool calls, delta 19.177 |
| `layers_ranking` | 0 tool calls, delta 0 | 0 tool calls, delta 0 |

O pior caso observado é enumeracao_ambientes no Sonnet: o prompt sai de 5.901 e chega a 25.078 tokens, mais que quadruplicando. O Haiku, na mesma pergunta, gasta um sétimo disso.

**O crescimento não vem do que as tools devolvem.** Em nenhum caso a saída das tools de consulta passou de 352 caracteres. O enumeracao_ambientes no Sonnet devolveu 131 caracteres e mesmo assim cresceu 19.177 tokens. Os tokens de raciocínio reportados também são pequenos diante disso (de 80 a 1.622). Ou seja, o custo está no
número de rodadas e no acúmulo da conversa, não no volume que cada tool insere.

> Isso orienta o notebook 2: o que reduz contexto é precisar de menos rodadas — chegar ao conjunto certo de entidades numa consulta só.